> **Cópia pública saneada.** Os dados de entrada não acompanham este repositório. Leia `docs/reprodutibilidade.md` e `docs/privacidade_e_dados.md` antes da execução. Notebooks de coleta dependem de rede; notebooks de tratamento escrevem somente em `data/`, que é ignorada pelo Git.

# Download dos dados

Este notebook será usado para baixar os arquivos identificados no catálogo do portal de dados abertos do BNDES.

A função deste notebook é operacional: baixar os arquivos CSV e PDF e armazená-los na pasta `data/raw`, sem ainda fazer limpeza, classificação verde ou análise estatística.

## 1. Preparação do ambiente

Neste bloco, carregamos as bibliotecas necessárias para localizar arquivos, consultar a internet, organizar tabelas e trabalhar com nomes padronizados.

As bibliotecas têm funções diferentes: `pathlib` organiza caminhos de arquivos, `json` lê o arquivo de configuração, `requests` faz o acesso aos links do BNDES, `pandas` organiza tabelas e `re`/`unicodedata` ajudam a limpar nomes de arquivos.

In [ ]:
from pathlib import Path
import json
import re
import unicodedata

import pandas as pd
import requests

## 2. Localização das pastas do projeto

Neste bloco, identificamos a pasta principal do projeto e definimos onde os arquivos brutos serão salvos.

A pasta `data/raw/csv` receberá as bases de dados em CSV. A pasta `data/raw/pdf` receberá os dicionários e documentos de apoio em PDF.

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_CSV_DIR = RAW_DIR / "csv"
RAW_PDF_DIR = RAW_DIR / "pdf"

RAW_CSV_DIR.mkdir(parents=True, exist_ok=True)
RAW_PDF_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

## 3. Carregamento das fontes do projeto

Neste bloco, carregamos o arquivo `configs/fontes_bndes.json`, que contém as bases iniciais do projeto.

Isso evita digitar manualmente os links em vários notebooks e reduz o risco de inconsistência entre as etapas da pesquisa.

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "configs" / "fontes_bndes.json"

with CONFIG_PATH.open("r", encoding="utf-8") as arquivo:
    fontes = json.load(arquivo)

pd.DataFrame(fontes["bases_iniciais"])

## 4. Configuração da API CKAN do BNDES

O portal de dados abertos do BNDES utiliza a estrutura CKAN, que permite consultar informações das bases por meio de uma API.

Neste bloco, definimos o endereço da API e criamos uma função auxiliar para fazer consultas ao portal.

Essa função será reutilizada para localizar os recursos disponíveis em cada base, recuperar seus links de download e organizar o catálogo que será usado na coleta dos arquivos.

In [ ]:
CKAN_API_BASE = fontes["portal_dados_abertos"].rstrip("/") + "/api/3/action"


def ckan_action(action, **params):
    url = f"{CKAN_API_BASE}/{action}"
    resposta = requests.get(url, params=params, timeout=60)
    resposta.raise_for_status()

    dados = resposta.json()

    if not dados.get("success"):
        raise RuntimeError(f"Erro na API CKAN: {dados}")

    return dados["result"]


CKAN_API_BASE

## 5. Reconstrução do catálogo de recursos

Neste bloco, consultamos novamente a API CKAN para reconstruir o catálogo dos recursos disponíveis nas quatro bases iniciais.

Embora o notebook anterior já tenha feito o inventário, aqui repetimos essa etapa de forma operacional para garantir que o notebook de download seja independente e possa ser executado sozinho.

O resultado será uma tabela com os arquivos CSV e PDF que serão baixados.

In [ ]:
catalogo_recursos = []

for base in fontes["bases_iniciais"]:
    nome_pacote = base["url"].rstrip("/").split("/")[-1]
    pacote = ckan_action("package_show", id=nome_pacote)

    for recurso in pacote["resources"]:
        catalogo_recursos.append({
            "base": base["nome"],
            "pacote": nome_pacote,
            "titulo_pacote": pacote["title"],
            "recurso_nome": recurso.get("name"),
            "format": recurso.get("format"),
            "id": recurso.get("id"),
            "url": recurso.get("url"),
            "datastore_active": recurso.get("datastore_active"),
            "last_modified": recurso.get("last_modified"),
        })

catalogo_recursos = pd.DataFrame(catalogo_recursos)

catalogo_recursos

## 6. Padronização dos nomes dos arquivos

Antes de baixar os arquivos, vamos criar nomes padronizados para salvá-los no projeto.

Essa etapa evita nomes longos, acentos, espaços e caracteres especiais. Também facilita a identificação posterior dos arquivos dentro da pasta `data/raw`.

Cada arquivo receberá um nome baseado na base de origem, no nome do recurso e no formato do arquivo.

In [ ]:
def limpar_nome_arquivo(texto):
    texto = str(texto).lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = texto.encode("ascii", "ignore").decode("ascii")
    texto = re.sub(r"[^a-z0-9]+", "_", texto)
    texto = texto.strip("_")
    return texto


catalogo_recursos["nome_arquivo"] = (
    catalogo_recursos["base"].apply(limpar_nome_arquivo)
    + "__"
    + catalogo_recursos["recurso_nome"].apply(limpar_nome_arquivo)
    + "."
    + catalogo_recursos["format"].str.lower()
)

catalogo_recursos[["base", "recurso_nome", "format", "nome_arquivo"]]

## 7. Definição dos caminhos de destino

Neste bloco, definimos em qual pasta cada arquivo será salvo.

Os arquivos CSV serão salvos em `data/raw/csv`, pois são as bases de dados usadas na análise quantitativa. Os arquivos PDF serão salvos em `data/raw/pdf`, pois funcionam como dicionários e documentação metodológica.

Ao final, teremos uma coluna com o caminho completo de destino de cada arquivo.

In [ ]:
def definir_caminho_destino(linha):
    formato = str(linha["format"]).upper()

    if formato == "CSV":
        return RAW_CSV_DIR / linha["nome_arquivo"]

    if formato == "PDF":
        return RAW_PDF_DIR / linha["nome_arquivo"]

    return RAW_DIR / linha["nome_arquivo"]


catalogo_recursos["caminho_destino"] = catalogo_recursos.apply(
    definir_caminho_destino,
    axis=1
)

catalogo_recursos[
    ["base", "recurso_nome", "format", "nome_arquivo", "caminho_destino"]
]

## 8. Função para baixar arquivos

Neste bloco, criamos uma função auxiliar para baixar arquivos a partir dos links do portal do BNDES.

A função recebe uma URL e um caminho de destino. Em seguida, faz o download em partes, grava o arquivo no computador e retorna informações básicas sobre o resultado.

Essa função será usada tanto para os arquivos CSV quanto para os arquivos PDF.

In [ ]:
def baixar_arquivo(url, caminho_destino):
    caminho_destino = Path(caminho_destino)
    caminho_destino.parent.mkdir(parents=True, exist_ok=True)

    resposta = requests.get(url, stream=True, timeout=120)
    resposta.raise_for_status()

    total_bytes = 0

    with caminho_destino.open("wb") as arquivo:
        for bloco in resposta.iter_content(chunk_size=1024 * 1024):
            if bloco:
                arquivo.write(bloco)
                total_bytes += len(bloco)

    return {
        "caminho": caminho_destino,
        "tamanho_mb": total_bytes / (1024 * 1024),
    }

## 9. Teste de download com um arquivo

Antes de baixar todos os arquivos, vamos testar a função com apenas um recurso.

Esse teste reduz o risco de baixar vários arquivos com erro de caminho, nome ou acesso. Se o primeiro download funcionar, seguimos para o download completo dos CSVs e PDFs.

In [ ]:
arquivo_teste = catalogo_recursos.iloc[0]

resultado_teste = baixar_arquivo(
    url=arquivo_teste["url"],
    caminho_destino=arquivo_teste["caminho_destino"]
)

resultado_teste

## 10. Download de todos os arquivos do catálogo

Neste bloco, baixamos todos os arquivos identificados no catálogo de recursos.

O procedimento percorre cada linha do catálogo, acessa o link original do portal do BNDES e salva o arquivo no caminho definido anteriormente.

Ao final, será criada uma tabela de controle com o status de cada download, o caminho salvo e o tamanho aproximado do arquivo.

In [ ]:
resultados_download = []

for _, recurso in catalogo_recursos.iterrows():
    caminho_destino = Path(recurso["caminho_destino"])

    if caminho_destino.exists():
        tamanho_mb = caminho_destino.stat().st_size / (1024 * 1024)

        resultados_download.append({
            "base": recurso["base"],
            "recurso_nome": recurso["recurso_nome"],
            "format": recurso["format"],
            "status": "ja_existia",
            "caminho": caminho_destino,
            "tamanho_mb": tamanho_mb,
        })

        continue

    resultado = baixar_arquivo(
        url=recurso["url"],
        caminho_destino=caminho_destino
    )

    resultados_download.append({
        "base": recurso["base"],
        "recurso_nome": recurso["recurso_nome"],
        "format": recurso["format"],
        "status": "baixado",
        "caminho": resultado["caminho"],
        "tamanho_mb": resultado["tamanho_mb"],
    })

resultados_download = pd.DataFrame(resultados_download)

resultados_download

## 11. Validação dos arquivos baixados

Neste bloco, verificamos se todos os arquivos informados na tabela de downloads realmente existem nas pastas do projeto.

Também calculamos um resumo por formato e status, o que permite confirmar quantos arquivos CSV e PDF foram baixados ou já estavam disponíveis.

Essa etapa é importante para garantir que a coleta foi concluída antes de avançar para armazenamento e leitura dos dados.

In [ ]:
resultados_download["arquivo_existe"] = resultados_download["caminho"].apply(
    lambda caminho: Path(caminho).exists()
)

resumo_download = (
    resultados_download
    .groupby(["format", "status", "arquivo_existe"], dropna=False)
    .agg(
        quantidade=("recurso_nome", "count"),
        tamanho_total_mb=("tamanho_mb", "sum")
    )
    .reset_index()
)

resumo_download

## 12. Salvamento do controle de downloads

Neste bloco, salvamos uma planilha com o controle dos arquivos baixados.

Esse arquivo registra a base de origem, o nome do recurso, o formato, o status do download, o caminho local e o tamanho aproximado de cada arquivo.

Esse controle é importante para documentação e replicabilidade da coleta.

In [ ]:
OUTPUT_TABLES_DIR = PROJECT_ROOT / "results" / "tables"
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

arquivo_controle_downloads = OUTPUT_TABLES_DIR / "controle_downloads_bndes.xlsx"

resultados_download.to_excel(
    arquivo_controle_downloads,
    index=False,
    sheet_name="Downloads"
)

arquivo_controle_downloads

## 13. Síntese da etapa de download

Nesta etapa, reconstruímos o catálogo dos recursos das quatro bases iniciais, padronizamos os nomes dos arquivos, definimos os caminhos de destino e realizamos o download dos arquivos brutos.

Ao final, os arquivos foram salvos em duas pastas: `data/raw/csv`, para as bases de dados, e `data/raw/pdf`, para os dicionários e documentação metodológica.

Também foi criada uma planilha de controle dos downloads, que registra o status, o caminho local e o tamanho de cada arquivo.

In [ ]:
print("Total de arquivos no catálogo:", len(catalogo_recursos))
print("Arquivos CSV baixados:", (resultados_download["format"].str.upper() == "CSV").sum())
print("Arquivos PDF baixados:", (resultados_download["format"].str.upper() == "PDF").sum())
print("Todos os arquivos existem:", resultados_download["arquivo_existe"].all())
print("Tamanho total aproximado (MB):", round(resultados_download["tamanho_mb"].sum(), 2))
print("Controle de downloads:", arquivo_controle_downloads)